# Data Pipeline: From Scratch

Feeding data to a GPU efficiently is one of the most critical parts of LLM training. If your data pipeline is slow, your expensive GPU sits idle.

In this tutorial, we will:
1. Build a **Manual Data Generator** to understand how to stream and batch data.
2. Implement a **Collate Function** from scratch to handle padding and tensor conversion.
3. Use PyTorch's `DataLoader` for multi-threaded loading.

## 1. Manual Data Generator

Imagine we have a massive dataset that doesn't fit in RAM. We need to read it one item at a time.

In [1]:
import time

# Simulate a large dataset
dummy_data = [f"Sample text {i}" for i in range(100)]

def data_generator(data, batch_size):
    batch = []
    for item in data:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

# Test it
for batch in data_generator(dummy_data, batch_size=4):
    print(batch)
    break # Just show one

['Sample text 0', 'Sample text 1', 'Sample text 2', 'Sample text 3']


## 2. Tokenization and Collation

Raw text isn't enough. We need to tokenize it and stack it into tensors. This is the job of the `collate_fn`.

In [2]:
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def manual_collate(batch_texts):
    # 1. Tokenize everything
    encoded = [tokenizer.encode(t) for t in batch_texts]
    
    # 2. Find max length in this batch
    max_len = max(len(t) for t in encoded)
    
    # 3. Pad sequences
    padded = []
    for seq in encoded:
        # Add padding tokens (pad_token_id) until max_len
        pad_len = max_len - len(seq)
        padded_seq = seq + [tokenizer.pad_token_id] * pad_len
        padded.append(padded_seq)
        
    # 4. Convert to Tensor
    return torch.tensor(padded)

# Test it
batch_texts = ["Hello world", "This is a longer sentence", "Short"]
batch_tensor = manual_collate(batch_texts)
print("Batch Tensor Shape:", batch_tensor.shape)
print(batch_tensor)

/Users/vukrosic/miniconda3/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Batch Tensor Shape: torch.Size([3, 5])
tensor([[19556,   905,     0,     0,     0],
        [ 1348,   314,   253,  2848,  6330],
        [20355,     0,     0,     0,     0]])


## 3. PyTorch DataLoader

Doing this manually in Python is slow because of the Global Interpreter Lock (GIL). PyTorch's `DataLoader` uses multiprocessing to do this in parallel.

In [4]:
from datasets import load_dataset
from torch.utils.data import DataLoader

# Load real dataset
dataset = load_dataset("HuggingFaceTB/smollm-corpus", "cosmopedia-v2", split="train", streaming=True)

def collate_fn(batch):
    texts = [item["text"] for item in batch]
    # AutoTokenizer handles padding/truncation much faster in Rust
    return tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")

loader = DataLoader(dataset, batch_size=4, collate_fn=collate_fn)

batch = next(iter(loader))
print("DataLoader Batch Shape:", batch['input_ids'].shape)

DataLoader Batch Shape: torch.Size([4, 512])
